# Posterior-predictive video viewer

Experimental-versus-synthetic viewer for the posterior-predictive video check of
`srm-and-sbi-monomer-dimer-alp`. One engine
(`srm_and_sbi_monomer_dimer_alp/posterior_predictive_video_runner.py`) renders a synthetic video
beside an experimental recording and persists both in one self-contained clip
(`*_Synthetic_Video.npz`: the two videos plus the full provenance). Two thin entry points call it:
the biology one (`Script_Bank/Analysis/SRM_AND_SBI_MONOMER_DIMER_ALP_Posterior_Predictive_Video.py`)
and the detector one (`..._DETECTOR_Posterior_Predictive_Video.py`); the clips share one format and
this notebook views either. It only **views** a persisted clip: it never runs a simulation, so
scrubbing, zooming and re-running cells never regenerate anything. A new trajectory means
re-running the engine, a deliberate step, optionally with another `--seed`.

**What the synthetic panel is.** The clip names its own synthetic source (`synth_label`), printed
below and used as the panel title. In the visual check at a declared configuration (the biology
entry point with `--declared-rds`), the motion comes from the declared reaction-diffusion values:
either one receptor total for every recording (the prior center) or one per recording, scaled from the
recording's opening spot count and capped by the prior range (the configuration cell prints the value
the clip used); the imaging is
the tagged `Nuisance_DLI` vector (`--nuisance-tag`, e.g. `REF`) plus the MET SCOPE camera values;
the labeling is the production labeling of the condition (the dye-count law at the declared
occupancy). Nothing is fitted to the recording: the comparison is a visual sanity check, read at
50 fps and through the pixel histograms of the static figure, not an estimate. With a MAP source
instead, the motion (biology) or the imaging (detector) comes from the recording's MAP estimate.

**How to run.** Pure viewer: it needs only `numpy`, `matplotlib` and `ipywidgets` (no project
package, no `MACHINE_PROFILE`, no ReaDDy). Render the clip with the entry point on the machine
that holds the data, copy the `*_Synthetic_Video.npz` here if needed, set `CLIP_PATH` in the
next code cell and launch `jupyter lab`. To compare two renders of the SAME recording (for
example the production render and one of its labeling arms), set `SECOND_CLIP_PATH` as well: the
second clip's synthetic video appears as a third panel, under the one window shared by all
panels, and the cell checks that both clips reference the same experimental frames.

**Color scaling.** All panels always share ONE window, mirroring the engine's static figure, so
identical intensities map to identical colors and a given frame looks the same in the scrubber, the
player and the figure drawn under the same mode (with a second clip, only `experimental` equals each
clip's figure, since the other modes span both synthetic clips), and the window is fixed over ALL
frames, so the brightness never changes from frame to frame. `NORM_MODE` sets what the window is. By
default (`percentile`) it is the whole-clip `[p_lower, p_upper]` of every clip shown at the editable
`NORM_PERCENTILES` pair, by default `(0, 99.99)`: the minimum to the p99.99, so the top 0.01 % of
pixels saturates and single hot pixels cannot dim the frames. `experimental` takes the experimental
clip's whole-clip min/max instead: the recording on its own scale, synthetic pixels outside it
saturating (above it as the brightest color, below it as black, like the background). `synthetic`
takes the whole-clip min/max of every synthetic clip shown, and experimental pixels outside it
saturate. `full` spans every clip shown, so nothing saturates, but the brightest pixel anywhere sets
the top, which makes the frames dim. The configuration cell prints the window and how many pixels of
each clip fall below and above it, and the scrubber and the player show the window in their titles. Not
offered: a per-frame window (recomputed for each displayed frame; it made the brightness jump between
frames, which is useless for a comparison over time) and per-panel independent scaling (identical
colors would then no longer mean identical intensities). The data stay 16-bit.

The check's documentation, the list of all its files and the order of a check are in
`Script_Bank/Analysis/SRM_AND_SBI_MONOMER_DIMER_ALP_Posterior_Predictive_Video.md`.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, interact
# Pure viewer: needs only numpy + matplotlib + ipywidgets. No project package, no
# MACHINE_PROFILE, no ReaDDy -- rendering happens in the engine on the data machine.

In [ ]:
# Point CLIP_PATH at a *_Synthetic_Video.npz written by the posterior-predictive video engine
# (either entry point). The clip is self-contained (experimental + synthetic + provenance), so any
# absolute path works.
CLIP_PATH = "<absolute path to ..._Synthetic_Video.npz>"
# Optional: a second *_Synthetic_Video.npz of the SAME recording (e.g. a labeling arm of CLIP_PATH's render).
# Its synthetic video is shown as a third panel under the same shared window; None shows two panels.
SECOND_CLIP_PATH = None

# Display color scaling (data stay 16-bit; this only sets how the magma colormap is scaled).
# Every panel ALWAYS shares ONE window, in every mode, so identical intensities map to identical
# colors -- mirroring the engine's static figure (a given frame looks identical in the scrubber, the
# player and the figure drawn under the same mode; with SECOND_CLIP_PATH only "experimental" equals
# each clip's figure) -- and the window is FIXED over all frames, so the brightness never changes
# from frame to frame. The mode only sets WHAT it is:
#   "percentile"   -> the whole-clip [p_lower, p_upper] of every clip shown at the NORM_PERCENTILES pair
#                     below (default; (0, 99.99): the minimum to the p99.99, so the top 0.01 % of pixels
#                     saturates and single hot pixels cannot dim the frames). Edit the pair to taste;
#                     0 <= lower < upper <= 100.
#   "experimental" -> the experimental clip's whole-clip [min, max]: the recording on its own scale;
#                     synthetic pixels outside it saturate (above as the brightest color, below as
#                     black, like the background).
#   "synthetic"    -> the whole-clip [min, max] of every synthetic clip shown (both, with
#                     SECOND_CLIP_PATH); experimental pixels outside it saturate.
#   "full"         -> the whole-clip [min, max] of every clip shown: nothing saturates; frames look dim
#                     because the brightest pixel anywhere sets the top.
# The cell prints the window and each clip's pixels below and above it; the scrubber and the player
# show the window in their titles. A per-frame window (recomputed for each displayed frame) is not
# offered: it made the brightness jump between frames, which is useless for a comparison over time.
NORM_MODE = "percentile"
NORM_PERCENTILES = (0.0, 99.99)      # (lower, upper) percentiles of the "percentile" window; used only by "percentile"

import json
d = np.load(CLIP_PATH, allow_pickle=False)
experimental, synth = d["experimental"], d["synth"]        # (n_frames, H, W) uint16
n_frames = int(d["n_frames"])
fps = round(1.0 / float(d["frame_time_seconds"]))
SYNTH_LABEL = str(d["synth_label"])                        # the synthetic panel's title, as on the figure

def _rows(pairs, per_row=3):
    return "\n".join("  " + "  ".join(pairs[j:j + per_row]) for j in range(0, len(pairs), per_row))

# --- identity of this clip ---
print(f"{str(d['workflow'])} entry point  |  kind={str(d['kind'])} cell={int(d['cell'])}  "
      f"{n_frames} frames at {fps} fps  |  seed={int(d['seed'])}  |  engine version {str(d['package_version'])}")
print(f"experimental: {str(d['experimental_tif'])}")
print(f"synthetic:    {SYNTH_LABEL}")
print(f"experimental {experimental.shape}  synth {synth.shape}")
print(f"ADU  experimental  min {int(experimental.min())} median {int(np.median(experimental))} max {int(experimental.max())}")
print(f"ADU  synth         min {int(synth.min())} median {int(np.median(synth))} max {int(synth.max())}")

# --- the optional second synthetic clip of the same recording (a third panel) ---
# PANELS lists every panel the scrubber and the player draw: the experimental frames, then each synthetic.
PANELS = [("EXPERIMENTAL", experimental), (SYNTH_LABEL, synth)]
if SECOND_CLIP_PATH:
    d2 = np.load(SECOND_CLIP_PATH, allow_pickle=False)
    if not np.array_equal(d2["experimental"], experimental):
        raise ValueError("SECOND_CLIP_PATH must be a clip of the same recording (its experimental frames differ)")
    synth2 = d2["synth"]
    SYNTH_LABEL_2 = str(d2["synth_label"])
    _lab2 = json.loads(str(d2["labeling_record_json"]))
    _r2 = {str(k): int(round(float(v))) for k, v in zip(d2["labeling_columns"], d2["labeling_row"])}
    print(f"second synthetic: {SYNTH_LABEL_2}  |  seed={int(d2['seed'])}  |  engine version {str(d2['package_version'])}")
    print(f"  labeling: {str(d2['condition'])} {_lab2['law']['description']}; "
          f"{_r2['n_labeled_subunits']} labeled subunits carrying {_r2['n_dyes']} dyes")
    print(f"ADU  synth (second) min {int(synth2.min())} median {int(np.median(synth2))} max {int(synth2.max())}")
    PANELS.append((SYNTH_LABEL_2, synth2))

def _num(v):
    """A number as the figure prints it (the engine's compact_number): at most four significant digits and
    three decimal places; below 0.01 three significant digits; 1000 or more a whole number."""
    v = float(v)
    if not np.isfinite(v):
        return str(v)
    if v == 0.0:
        return "0"
    if abs(v) < 0.01:
        return f"{v:.3g}"
    if abs(v) >= 1000.0:
        return f"{float(f'{v:.4g}'):.0f}"
    s = f"{v:.4g}"
    return f"{v:.3f}" if ("e" in s or ("." in s and len(s.split(".")[1]) > 3)) else s

# --- provenance, as the static figure states it ---
# Imaging: the eleven values the render used (six emitter parameters + five SCOPE camera values).
_tag = str(d["nuisance_tag"])
print(f"imaging ({str(d['imaging_desc'])}):" if not _tag else f"imaging (tag {_tag}; {str(d['imaging_desc'])}):")
print(_rows([f"{k}={_num(v)}" for k, v in zip(d["imaging_keys"], d["imaging_physical"])]))
# Reaction-diffusion: the eleven values the motion came from -- declared (configuration file,
# name and SHA-256) or the recording's MAP estimate (biology), or the marginalized nuisance draw (detector).
_rds_src = str(d["rds_source"])
_rds = json.loads(str(d["rds_record_json"]))
_scn = _rds.get("scenario", {})
_head = f"reaction-diffusion ({_rds_src}"
if _scn.get("name"):
    _head += f": {_scn['name']}, {_rds['path']}, sha256 {_rds['sha256'][:12]}"
print(_head + "):")
_out = set(str(k) for k in d["rds_outside_prior"])
print(_rows([(f"{k}={float(v):.0f}" if str(k) == "count_total" else f"{k}={_num(v)}") + ("*" if str(k) in _out else "")
             for k, v in zip(d["rds_keys"], d["rds_provenance"])]))
if _out:
    print(f"  (* outside the biology prior: {sorted(_out)})")
# Labeling: the production labeling this render used, and the receptors it actually started with.
_lab = json.loads(str(d["labeling_record_json"]))
_occ = _lab["occupancy"]
_occ = ", ".join(f"{k}={_num(v)}" for k, v in _occ.items()) if isinstance(_occ, dict) else _num(_occ)
_vis = ", ".join(f"{k} {_num(v)}" for k, v in _lab["visible_per_subunit"].items())
# A labeling arm redrew the dye counts on its source clip's labeled subunits, as the figure says.
_kept = "; the source's labeled subunits kept" if _lab.get("arm") else ""
print(f"labeling: {str(d['condition'])} {_lab['law']['description']}, occupancy {_occ} "
      f"({_lab['occupancy_source']}{_kept}); visible per subunit {_vis}")
_r = {str(k): int(round(float(v))) for k, v in zip(d["labeling_columns"], d["labeling_row"])}
_n_mono, _n_dim = _r["monomers_0"], _r["dimers_0"]
_v_mono, _v_dim = _r["monomers_visible_0"], _r["dimers_visible_0"]
print("initial receptors (frame 0, this render):\n"
      f"  {_r['n_subunits']} subunits = {_n_mono} monomers + {_n_dim} dimers ({_n_mono + _n_dim} receptors)\n"
      f"  labeled: {_r['n_labeled_subunits']} subunits carrying {_r['n_dyes']} dyes\n"
      f"  visible: {_v_mono} of {_n_mono} monomers + {_v_dim} of {_n_dim} dimers "
      f"({_r['dimers_two_labeled_0']} with both subunits labeled) = {_v_mono + _v_dim} spots")

# The four windows, each computed once over ALL frames and shared by every panel. NORM_MODES repeats
# the engine's DISPLAY_NORMS (this viewer imports no project package); a test keeps the two equal.
NORM_MODES = ("percentile", "experimental", "synthetic", "full")
if NORM_MODE not in NORM_MODES:
    raise ValueError(f"NORM_MODE must be one of {NORM_MODES}, not {NORM_MODE!r}")
_P_LO, _P_HI = (float(v) for v in NORM_PERCENTILES)
if not (0.0 <= _P_LO < _P_HI <= 100.0):
    raise ValueError(f"NORM_PERCENTILES must satisfy 0 <= lower < upper <= 100, not {NORM_PERCENTILES!r}")
_STACKS = [s for _, s in PANELS]                   # every clip shown: the experimental frames, then each synthetic
_SYNTHS = _STACKS[1:]
_WINDOWS = {
    "experimental": (float(experimental.min()), float(experimental.max())),
    "synthetic": (float(min(s.min() for s in _SYNTHS)), float(max(s.max() for s in _SYNTHS))),
    "full": (float(min(s.min() for s in _STACKS)), float(max(s.max() for s in _STACKS))),
    "percentile": (float(min(np.percentile(s, _P_LO) for s in _STACKS)),
                   float(max(np.percentile(s, _P_HI) for s in _STACKS))),
}
_WINDOW_DESC = {"experimental": "the experimental clip's whole-clip [min, max]",
                "synthetic": "the whole-clip [min, max] of every synthetic clip shown",
                "full": "the whole-clip [min, max] of every clip shown",
                "percentile": f"the whole-clip [p{_P_LO:g}, p{_P_HI:g}] of every clip shown"}

def clim(*frames):
    """ONE shared (vmin, vmax) for ALL panels, fixed over all frames: the NORM_MODE window. Identical
    intensities map to identical colors, mirroring the engine. (The frames are accepted so a caller can
    pass the displayed ones; the window never depends on them.)"""
    return _WINDOWS[NORM_MODE]

# The window, and each clip's pixels outside it: above it they show as the brightest color, below it
# as black, like the background.
_lo, _hi = clim()
_names = ("experimental", "synthetic", "second synthetic")
print(f"NORM_MODE={NORM_MODE}  window [{_lo:.0f}, {_hi:.0f}] ADU: {_WINDOW_DESC[NORM_MODE]}, shared by "
      f"every panel and fixed over all frames")
print("outside it: " + "; ".join(f"{name} {int(np.count_nonzero(s < _lo))} below, "
                                 f"{int(np.count_nonzero(s > _hi))} above"
                                 for name, s in zip(_names, _STACKS)))


In [ ]:
# Scrub frames and zoom a shared region-of-interest into ALL panels together.
# zoom = 1 shows the full frame; higher zoom crops tighter around (center x, center y).
H, W = experimental.shape[1], experimental.shape[2]

def _roi(frame, cx, cy, zoom):
    half_y, half_x = int(H / (2 * zoom)), int(W / (2 * zoom))
    y0, y1 = max(0, cy - half_y), min(H, cy + half_y)
    x0, x1 = max(0, cx - half_x), min(W, cx + half_x)
    return frame[y0:y1, x0:x1], (x0, x1, y0, y1)

def show(frame, cx, cy, zoom):
    crops = [_roi(stack[frame], cx, cy, zoom) for _, stack in PANELS]
    ext = crops[0][1]
    vmin, vmax = clim(*(c for c, _ in crops))        # ONE shared window over all panels
    fig, ax = plt.subplots(1, len(PANELS), figsize=(6 * len(PANELS), 6))
    for a, (crop, _), (title, _) in zip(ax, crops, PANELS):
        a.imshow(crop, cmap="magma", origin="lower", interpolation="none",
                 vmin=vmin, vmax=vmax, extent=ext)
        a.set_title(f"{title}   frame {frame}/{n_frames - 1}   zoom x{zoom:g}", fontsize=9)
    fig.suptitle(f"display window [{vmin:.0f}, {vmax:.0f}] ADU ({NORM_MODE})", fontsize=10)
    plt.tight_layout(); plt.show()

interact(
    show,
    frame=IntSlider(min=0, max=n_frames - 1, step=1, value=n_frames // 2, description="frame"),
    cx=IntSlider(min=0, max=W - 1, step=1, value=W // 2, description="center x"),
    cy=IntSlider(min=0, max=H - 1, step=1, value=H // 2, description="center y"),
    zoom=FloatSlider(min=1.0, max=8.0, step=0.5, value=1.0, description="zoom"),
);

## Playback (real time)

Play the clips side by side at the native frame rate (50 fps for the production tier), using the
**same shared color scaling as the scrubber** (all panels on one window fixed over all frames, so a
given frame looks the same here, in the scrubber and in the static figure drawn under the same mode). Set `PLAY_ZOOM` (and the
center) to play a **cropped region** and check whether experimental and synthetic coincide locally.
The player embeds every frame losslessly (PNG frames), so a long clip builds a large widget: the
embed limit is raised below, and the cell reports the player's size and confirms that every frame
was embedded. Raise `PLAY_EVERY` only to lighten the build (playback stays real time).


In [ ]:
import matplotlib as mpl
from matplotlib import animation
from IPython.display import HTML

mpl.rcParams["animation.embed_limit"] = 768   # MB; a 1000-frame 2-panel player is ~400-490 MB under "percentile"
                                               # (lossless PNG frames of camera noise barely compress, ~350 KB per
                                               # two-panel frame, plus a third for the base64 embedding); a third
                                               # panel adds about half
PLAY_EVERY = 1                                 # stride: raise (2, 5, ...) to shrink the player; stays real-time

# Zoom for the player (same ROI convention as the scrubber): PLAY_ZOOM = 1 plays the full frame;
# raise it and set the center to play a cropped region and inspect local agreement.
PLAY_ZOOM = 1.0
PLAY_CX, PLAY_CY = W // 2, H // 2

def _pcrop(frame):
    return _roi(frame, PLAY_CX, PLAY_CY, PLAY_ZOOM)[0]

idx = list(range(0, n_frames, PLAY_EVERY))
fig, ax = plt.subplots(1, len(PANELS), figsize=(4.5 * len(PANELS), 4.6))
crops0 = [_pcrop(stack[0]) for _, stack in PANELS]
vmin, vmax = clim(*crops0)                         # ONE shared window over all panels
panels = []
for a, c0, (title, _) in zip(ax, crops0, PANELS):
    im = a.imshow(c0, cmap="magma", origin="lower", interpolation="none", vmin=vmin, vmax=vmax)
    a.set_title(title, fontsize=9); a.set_xticks([]); a.set_yticks([]); panels.append(im)
plt.close(fig)   # suppress the static duplicate; the player below renders it

def _update(i):
    crops = [_pcrop(stack[i]) for _, stack in PANELS]
    vmin, vmax = clim(*crops)                       # shared across all panels, fixed over all frames
    for im, cf in zip(panels, crops):
        im.set_data(cf); im.set_clim(vmin, vmax)
    fig.suptitle(f"frame {i}/{n_frames - 1}   ({fps} fps)   zoom x{PLAY_ZOOM:g}   "
                 f"window [{vmin:.0f}, {vmax:.0f}] ADU ({NORM_MODE})")
    return panels

anim = animation.FuncAnimation(fig, _update, frames=idx, interval=1000.0 * PLAY_EVERY / fps, blit=False)
player = anim.to_jshtml()
# Lossless PNG frames; every frame must be present -- matplotlib truncates silently above the embed limit.
_n_embedded = player.count("data:image/png;base64")
print(f"player: {len(player) / 1e6:.1f} MB, {_n_embedded} of {len(idx)} frames embedded"
      + ("" if _n_embedded == len(idx) else "  <-- TRUNCATED: raise animation.embed_limit or PLAY_EVERY"))
HTML(player)
